# Study 07: Statistical Significance Tests\n**Goal:** Bootstrap resampling to determine if ensemble outperforms individual models significantly.

## 1. Setup

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')
print('Setup complete')

## 2. Load Per-Sample Dice

In [ ]:
with open(Path.cwd().parent/'results'/'per_sample_dice.json') as f:
    per_sample = json.load(f)
print(f'Loaded {len(next(iter(per_sample.values())))} samples per model')
for name, dices in per_sample.items():
    print(f'  {name}: mean={np.mean(dices):.4f}, std={np.std(dices):.4f}')

## 3. Bootstrap Resampling

In [ ]:
N_BOOTSTRAP = 10000
rng = np.random.RandomState(42)
n = len(per_sample[list(per_sample.keys())[0]])

bootstrap_means = {name: [] for name in per_sample}
for _ in range(N_BOOTSTRAP):
    idx = rng.randint(0, n, n)
    for name, dices in per_sample.items():
        bootstrap_means[name].append(np.mean([dices[i] for i in idx]))

# Compute p-values
comparisons = [('ensemble', 'best_model'), ('ensemble', 'member_0'),
               ('ensemble', 'member_1'), ('ensemble', 'member_2')]
p_values = {}
for a, b in comparisons:
    diff = np.array(bootstrap_means[a]) - np.array(bootstrap_means[b])
    p = np.mean(diff <= 0) * 2  # two-sided
    p_values[f'{a}_vs_{b}'] = {'diff_mean': float(np.mean(diff)),
                                'diff_std': float(np.std(diff)),
                                'p_value': float(p),
                                'significant': p < 0.05}
    sig = 'SIGNIFICANT' if p < 0.05 else 'not significant'
    print(f'{a} vs {b}: diff={np.mean(diff):.4f}\u00b1{np.std(diff):.4f}, p={p:.4f} ({sig})')

with open(Path.cwd().parent/'results'/'statistical_tests.json', 'w') as f:
    json.dump(p_values, f, indent=2)

## 4. Bootstrap Distribution Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, (a, b) in enumerate(comparisons):
    ax = axes[i//2, i%2]
    diff = np.array(bootstrap_means[a]) - np.array(bootstrap_means[b])
    ax.hist(diff, bins=80, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(0, color='red', ls='--', lw=2, label='No difference')
    ax.axvline(np.mean(diff), color='green', ls='-', lw=2, label=f'Mean={np.mean(diff):.4f}')
    ax.set_title(f'{a} vs {b}: p={p_values[f"{a}_vs_{b}"]["p_value"]:.4f}')
    ax.set_xlabel('Dice difference'); ax.set_ylabel('Count'); ax.legend()
plt.tight_layout()
plt.savefig(Path.cwd().parent/'figures'/'bootstrap_significance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Key Findings

In [ ]:
print('='*60)
print('STATISTICAL SIGNIFICANCE SUMMARY')
print('='*60)
for comp, res in p_values.items():
    sig = 'YES' if res['significant'] else 'NO'
    print(f'{comp}: p={res["p_value"]:.4f}, significant={sig}')
print('\nConclusion: Ensemble significantly outperforms' if any(v["significant"] for v in p_values.values()) else '')